In [8]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("profiling").config("spark.sql.ansi.enabled", "false").config("spark.driver.memory", "16g").getOrCreate()
df = spark.read.parquet("../data/cleaned/4c_eea_co2_emissions_from_passenger_cars-001.parquet")

df = df.drop("registrations")
df.show(5, truncate=False)

+---+-------------------------------+-----------+------------------------------------------+-----------------------+-------+-------------+--------------------------+-------------------------+---------------------+-----------------+-----------------------------------+
|geo|Geopolitical entity (reporting)|TIME_PERIOD|manufacturer_name_eu_standard_denomination|commercial_name        |variant|Motor energy |mass_in_running_order (kg)|co2_emissions_WLTP (g/km)|engine_capacity (cm3)|engine_power (KW)|electric_energy_consumption (Wh/km)|
+---+-------------------------------+-----------+------------------------------------------+-----------------------+-------+-------------+--------------------------+-------------------------+---------------------+-----------------+-----------------------------------+
|DE |Germany                        |2023       |VOLVO                                     |XC40                   |XZBW   |Petrol hybrid|1812                      |48                       |1477 

Get the number of unique combinations of <'commercial_name', 'engine_capacity (cm3)', and 'engine_power (KW)'> for the consumer for each <'TIME_PERIOD', 'geo', 'Geopolitical entity (reporting)', 'Motor energy'>.

In [9]:
unique_consumer_choices_counts = (
    df
    .groupBy(
        "TIME_PERIOD",
        "geo",
        "Geopolitical entity (reporting)",
        "Motor energy"
    )
    .agg(
        F.countDistinct(
            F.struct(
                "commercial_name",
                "engine_capacity (cm3)",
                "engine_power (KW)"
            )
        ).alias("unique_choices")
    )
    .orderBy(
        "TIME_PERIOD",
        "geo",
        "Motor energy"
    )
)

unique_consumer_choices_counts.show(20, truncate=False)

+-----------+---+-------------------------------+--------------------------+--------------+
|TIME_PERIOD|geo|Geopolitical entity (reporting)|Motor energy              |unique_choices|
+-----------+---+-------------------------------+--------------------------+--------------+
|2014       |AT |Austria                        |Alternative/Other         |26            |
|2014       |AT |Austria                        |Diesel (excluding hybrids)|1523          |
|2014       |AT |Austria                        |Diesel hybrid             |2             |
|2014       |AT |Austria                        |Electricity               |16            |
|2014       |AT |Austria                        |Petrol (excluding hybrids)|1419          |
|2014       |AT |Austria                        |Petrol hybrid             |14            |
|2014       |BE |Belgium                        |Alternative/Other         |41            |
|2014       |BE |Belgium                        |Diesel (excluding hybrids)|1387